# E12 · Reporting delays (simulation)

Protocol addendum: [`protocolo_revision.md`](../protocolo_revision.md). Written after E1–E8, in response to external feedback; every result here is exploratory. Negative changes favour the first arm; intervals are 95% whole-season bootstrap intervals (20,000 draws, seeds paired).

The surveillance file has no notification dates, so delays are assumed. One multinomial draw per district-week splits the final count into cases reported 0, 1, 2 and 3 weeks after the week closes; at each origin, the recent case history, the persistence offset, the persistence term of the count blend and calibrated persistence use the counts reported by that origin. Outcomes, thresholds and evaluation keep the final counts. Scenarios (completeness after 0, 1, 2 weeks): moderate 60/85/95%, severe 30/60/85%.

In [1]:
import os
import sys
from pathlib import Path

os.environ.update({"DENGUEKAY_XGB_DEVICE": "cpu", "DENGUEKAY_XGB_NJOBS": "2", "OMP_NUM_THREADS": "2",
                   "OPENBLAS_NUM_THREADS": "1", "VECLIB_MAXIMUM_THREADS": "1", "MLFLOW_DISABLE_AGENT_HINT": "1"})
STUDY = Path.cwd().parent
sys.path.insert(0, str(STUDY / "code"))

import numpy as np
import pandas as pd

import common as c

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)
import matplotlib.pyplot as plt

import experiments as ex

RES = c.OUTPUT / "resultados"
FIG = STUDY / "figures"
OTHER = tuple(s for s in c.SEASONS if s not in c.DECISION_SEASONS)
read_base = lambda name: pd.read_csv(c.OUTPUT / name, dtype={"ubigeo": str}, parse_dates=["semana_inicio", "origen_cierre"])

import reporting_delays as rd

In [2]:
inputs, _ = c.load_inputs(validate=False)
final = {"climate": c.read_arm("b3_obs", columns=ex.PRED_COLUMNS), "no climate": c.read_arm("historia_geo", columns=ex.PRED_COLUMNS)}
final_base = read_base("baselines.csv.gz")
sse_final = {k: ex.sse(v) for k, v in final.items()}
sse_final["persistence"] = ex.sse(final_base, "persistencia_prob")
KEY = ["semilla", "bloque", "horizonte", "ubigeo", "semana_inicio"]
OUTCOME = ["casos_Dengue", "brote", "umbral_brote_casos"]

In [3]:
rows, checks, count_rows = [], [], []
for scenario in rd.SCENARIOS:
    folder = rd.OUT_DIR / scenario
    arms = {"climate": c.read_arm("b3_obs", columns=ex.PRED_COLUMNS, folder=folder),
            "no climate": c.read_arm("historia_geo", columns=ex.PRED_COLUMNS, folder=folder)}
    merged = arms["climate"].merge(final["climate"], on=KEY, suffixes=("", "_final"), validate="one_to_one")
    same = all(merged[col].equals(merged[f"{col}_final"]) for col in OUTCOME)
    ev = ex.seasons_only(merged)
    checks.append({"scenario": scenario, "rows": len(merged), "outcomes_identical": same,
                   "origin_count_reported_%": 100 * ev.persistencia.sum() / ev.persistencia_final.sum()})
    support = arms["climate"][arms["climate"].semilla.eq(17)][c.archive.OBSERVATIONS].reset_index(drop=True)
    base = c.baselines(support, inputs["etiquetas"])
    s = {k: ex.sse(v) for k, v in arms.items()}
    s["persistence"] = ex.sse(base, "persistencia_prob")
    for a, b in (("climate", "no climate"), ("climate", "persistence"), ("no climate", "persistence")):
        rows.append(ex.contrast(s[a], s[b]).assign(scenario=scenario, comparison=f"{a} vs {b}"))
    for k in ("no climate", "climate", "persistence"):
        rows.append(ex.contrast(s[k], sse_final[k]).assign(scenario=scenario, comparison=f"{k}: delays vs final counts"))
    sa, sb = ex.sae(arms["climate"]), ex.sae(arms["no climate"])
    count_rows.append(ex.contrast(sa, sb, value="sae").assign(scenario=scenario, comparison="climate vs no climate (MAE)"))
checks = pd.DataFrame(checks)
assert checks.outcomes_identical.all()
checks.round(2)

,scenario,rows,outcomes_identical,origin_count_reported_%
0,moderado,974025,True,60.01
1,severo,974025,True,29.80


In [4]:
reference = [ex.contrast(sse_final["climate"], sse_final["no climate"]).assign(scenario="final counts", comparison="climate vs no climate")]
delays = pd.concat(reference + rows)[["scenario", "comparison", "horizonte", "brier_a", "brier_b", "cambio_agrupado_%",
                                     "ic95_agrupado", "cambio_igual_peso_%", "temporadas_a_mejor"]]
delays.to_csv(RES / "e12_retrasos.csv", index=False)
delays.round(4)

,scenario,comparison,horizonte,brier_a,brier_b,cambio_agrupado_%,ic95_agrupado,cambio_igual_peso_%,temporadas_a_mejor
0,final counts,climate vs no climate,2,0.0282,0.0283,-0.4300,"(-1.46, 0.37)",-1.3127,10
1,final counts,climate vs no climate,3,0.0322,0.0323,-0.2760,"(-1.89, 0.87)",-1.3944,10
2,final counts,climate vs no climate,4,0.0356,0.0356,0.0416,"(-2.31, 1.7)",-1.1533,10
0,moderado,climate vs no climate,2,0.0292,0.0292,-0.3087,"(-1.73, 0.74)",-0.8423,10
1,moderado,climate vs no climate,3,0.0331,0.0332,-0.3577,"(-2.33, 0.95)",-1.1770,10
2,moderado,climate vs no climate,4,0.0361,0.0363,-0.4122,"(-3.19, 1.52)",-1.5440,8
0,moderado,climate vs persistence,2,0.0292,0.0310,-5.9554,"(-10.42, 0.41)",4.9267,9
1,moderado,climate vs persistence,3,0.0331,0.0355,-6.6532,"(-11.74, 1.22)",5.2232,8
2,moderado,climate vs persistence,4,0.0361,0.0396,-8.9320,"(-14.62, 0.13)",6.4511,8
0,moderado,no climate vs persistence,2,0.0292,0.0310,-5.7118,"(-9.97, 1.02)",6.4059,9


In [5]:
mae = pd.concat(count_rows).rename(columns={"brier_a": "mae_a", "brier_b": "mae_b"})
mae = mae[["scenario", "comparison", "horizonte", "mae_a", "mae_b", "cambio_agrupado_%", "ic95_agrupado", "temporadas_a_mejor"]]
mae.to_csv(RES / "e12_retrasos_conteos.csv", index=False)
mae.round(3)

,scenario,comparison,horizonte,mae_a,mae_b,cambio_agrupado_%,ic95_agrupado,temporadas_a_mejor
0,moderado,climate vs no climate (MAE),2,1.772,1.821,-2.721,"(-5.26, 0.61)",9
1,moderado,climate vs no climate (MAE),3,2.136,2.164,-1.257,"(-5.08, 1.1)",9
2,moderado,climate vs no climate (MAE),4,2.481,2.573,-3.591,"(-11.96, 0.96)",10
0,severo,climate vs no climate (MAE),2,1.873,2.032,-7.873,"(-14.03, -1.92)",11
1,severo,climate vs no climate (MAE),3,2.234,2.366,-5.542,"(-10.53, 0.49)",6
2,severo,climate vs no climate (MAE),4,2.487,2.611,-4.745,"(-11.99, 0.85)",8


## Reading

- Outcomes are identical to the final-count arms; at issue time the origin week showed about 60% (moderate) and 30% (severe) of its final count.
- Climate effect: final counts −0.43%, −0.28% and +0.04%; moderate −0.31%, −0.36% and −0.41%; severe −0.38%, −0.66% and −0.90%. Every interval includes zero. Delays moved the four-week estimate towards climate by 0.5 (moderate) and 0.9 (severe) points.
- Delays worsened the model without climate by 1.8–3.1% (moderate) and 5.5–9.5% (severe), and calibrated persistence by 5.9–8.3% and 17.0–29.1%. Under severe delays both models beat persistence by 14–16% at every horizon, with intervals that exclude zero.
- Count MAE, climate vs no climate: −1.3% to −3.6% (moderate) and −4.7% to −7.9% (severe); only the two-week change under severe delays has an interval that excludes zero.